# Project 76 — Local Distillation Lab
## Teacher → Compressed Student → Quality Comparison

**Stack:** LangChain · Ollama · pandas · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama pandas

## Step 1 — Define Knowledge Domains

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import time, pandas as pd, json
from pathlib import Path

teacher = ChatOllama(model="qwen3:8b", temperature=0.3)

domains = [
    "Explain how a hash table works, including collision resolution strategies",
    "Describe the difference between TCP and UDP with real-world examples",
    "Explain garbage collection in programming languages",
    "What is a deadlock and how do you prevent it?",
    "Explain the CAP theorem with database examples",
    "How does TLS/SSL encryption work?",
    "Explain ACID properties in databases",
    "What is eventual consistency and when is it acceptable?",
]
print(f"Knowledge domains: {len(domains)} topics")

## Step 2 — Generate Teacher Explanations

In [ ]:
teacher_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert teacher. Provide a thorough explanation with examples, "
     "analogies, and edge cases. Be comprehensive."),
    ("human", "{topic}")
])
teacher_chain = teacher_prompt | teacher | StrOutputParser()

teacher_outputs = []
for topic in domains:
    start = time.time()
    explanation = teacher_chain.invoke({"topic": topic})
    elapsed = time.time() - start
    teacher_outputs.append({
        "topic": topic[:50], "output": explanation,
        "chars": len(explanation), "words": len(explanation.split()),
        "time_s": round(elapsed, 2),
    })
    print(f"  {topic[:45]}... → {len(explanation)} chars ({elapsed:.1f}s)")

print(f"\nTeacher avg: {sum(t['chars'] for t in teacher_outputs)//len(teacher_outputs)} chars/topic")

## Step 3 — Distill to Student Format

In [ ]:
compress_prompt = ChatPromptTemplate.from_messages([
    ("system", "Compress this teacher explanation into a concise student version. "
     "Rules:\n- Keep ALL key facts, definitions, and numbers\n"
     "- Remove examples, analogies, and verbose explanations\n"
     "- Target 30% of original length\n- Use bullet points"),
    ("human", "Teacher explanation:\n{teacher}")
])
compress_chain = compress_prompt | teacher | StrOutputParser()

distilled = []
for t_out in teacher_outputs:
    start = time.time()
    student = compress_chain.invoke({"teacher": t_out["output"]})
    elapsed = time.time() - start
    ratio = len(student) / max(len(t_out["output"]), 1)
    distilled.append({
        "topic": t_out["topic"],
        "teacher_output": t_out["output"],
        "student_output": student,
        "teacher_chars": t_out["chars"],
        "student_chars": len(student),
        "compression_ratio": round(ratio, 2),
        "time_s": round(elapsed, 2),
    })
    print(f"  {t_out['topic'][:40]}... {t_out['chars']}→{len(student)} chars (ratio={ratio:.0%})")

Path("sample_data").mkdir(exist_ok=True)
with open("sample_data/distillation_data.json", "w") as f:
    json.dump(distilled, f, indent=2, default=str)

## Step 4 — Quality Comparison

In [ ]:
from pydantic import BaseModel, Field

class QualityScore(BaseModel):
    completeness: float = Field(ge=0, le=1, description="Are all key facts preserved?")
    accuracy: float = Field(ge=0, le=1, description="Is everything correct?")
    clarity: float = Field(ge=0, le=1, description="Is it easy to understand?")

judge = teacher.with_structured_output(QualityScore)

rows = []
for d in distilled:
    score = judge.invoke(
        f"Compare student vs teacher explanation.\n\n"
        f"Teacher: {d['teacher_output'][:500]}\n\n"
        f"Student: {d['student_output'][:500]}\n\n"
        f"Rate the student version."
    )
    rows.append({
        "topic": d["topic"],
        "compression": d["compression_ratio"],
        "completeness": score.completeness,
        "accuracy": score.accuracy,
        "clarity": score.clarity,
        "avg_score": round((score.completeness + score.accuracy + score.clarity) / 3, 2),
    })

qdf = pd.DataFrame(rows)
print("DISTILLATION QUALITY")
print("=" * 60)
print(qdf.to_string(index=False))
print(f"\nAvg compression: {qdf['compression'].mean():.0%}")
print(f"Avg quality:     {qdf['avg_score'].mean():.0%}")

## What You Learned
- **Teacher-student distillation** pipeline
- **Compression with fact preservation** constraints
- **Quality scoring** of compressed outputs
- **Dataset generation** for student model training